# S2.13 — Repartition vs Coalesce
**Date completed:** September 2026  
**Status:** In Progress  
**Interview covered:** Q17 — repartition() vs coalesce()

In [0]:
# ============================================================
# Cell 2 — Repartition vs Coalesce: Practical Comparison
#
# Goal:
# 1. Inspect how repartition and coalesce differ.
# 2. Compare the same target partition count.
# 3. Understand shuffle versus narrow dependency.
# ============================================================

import pyspark.sql.functions as F
import time

# ------------------------------------------------------------
# Step 1 — Create dataset
# ------------------------------------------------------------

df = spark.range(0, 5000000)

df = df.withColumn(
    "value",
    (F.col("id") % 1000).cast("double")
)

# Observe initial non-empty partitions
starting_partitions = (
    df.select(F.spark_partition_id().alias("partition_id"))
      .distinct()
      .count()
)

print(f"Starting partitions: {starting_partitions}")


# ------------------------------------------------------------
# Step 2 — Repartition to 2
# A shuffle is introduced to redistribute data.
# ------------------------------------------------------------

print("\n=== TEST 1: repartition(2) ===")

df_rep = df.repartition(2)

# Inspect physical plan
df_rep.explain("formatted")

start = time.perf_counter()

rep_result = df_rep.agg(
    F.sum("value").alias("total_value")
).collect()

rep_time = time.perf_counter() - start

print(f"Result: {rep_result}")
print(f"Time: {rep_time:.4f} seconds")


# ------------------------------------------------------------
# Step 3 — Coalesce to 2
# Uses narrow dependency rather than a full shuffle.
# ------------------------------------------------------------

print("\n=== TEST 2: coalesce(2) ===")

df_coal = df.coalesce(2)

# Inspect physical plan
df_coal.explain("formatted")

start = time.perf_counter()

coal_result = df_coal.agg(
    F.sum("value").alias("total_value")
).collect()

coal_time = time.perf_counter() - start

print(f"Result: {coal_result}")
print(f"Time: {coal_time:.4f} seconds")


# ------------------------------------------------------------
# Step 4 — Compare results
# ------------------------------------------------------------

print("\n=== FINAL COMPARISON ===")

print(f"repartition(2): {rep_time:.4f} seconds")
print(f"coalesce(2)  : {coal_time:.4f} seconds")

## Key Takeaways — S2.13 Repartition vs Coalesce

## Core Difference — Proved with EXPLAIN
| | repartition() | coalesce() |
|--|--------------|------------|
| Shuffle | YES — PhotonShuffleExchange | NO — just merges |
| Direction | Increase OR decrease | Decrease ONLY |
| Cost | Expensive — network transfer | Cheap — local merge |
| Photon | Fully supported | Not supported (JVM fallback) |
| Speed (5M rows) | 0.5593s | 0.4498s ✅ faster |

## Key Rule

Need to INCREASE partitions → repartition() only option
Need to DECREASE partitions → coalesce() always preferred


## EXPLAIN proof
repartition → PhotonShuffleExchangeSink + PhotonShuffleExchangeSource
coalesce    → just "Coalesce" node — no shuffle operators

## When to use each — Real RetailPulse scenarios
**repartition:**
- After reading small CSV → increase before heavy transformation
- Repartition by column before join → df.repartition(8, col("city"))
- Even data distribution needed

**coalesce:**
- Before writing Gold layer → reduce to fewer clean files
- After groupBy on small result → avoid 200 tiny files
- df.groupBy("city").agg(...).coalesce(1).write.parquet(path)

## The lesson
Avoiding shuffle beats Photon acceleration.
No network transfer > C++ engine optimisation.

# S2.13 — repartition() vs coalesce()
### Spark Partition Management and Performance Optimisation

---

## 1. What Are repartition() and coalesce()?

Both functions control the number of partitions in a Spark DataFrame.

However, they use different execution strategies.

| Feature | repartition(n) | coalesce(n) |
|---|---|---|
| Purpose | Redistribute data | Reduce partitions |
| Increase partitions | Yes | No, not beyond current count |
| Decrease partitions | Yes | Yes |
| Full shuffle | Yes | Normally no |
| Network redistribution | Yes, potentially | Normally avoids full redistribution |
| Performance cost | Additional shuffle overhead | Usually lower redistribution overhead |
| Common use | Improve parallelism and redistribute data | Reduce unnecessary partitions |

---

## 2. repartition(n) — Redistribute Data

Syntax:

```python
df.repartition(8)
```

### How it works

Assume our DataFrame currently has 4 partitions.

```text
Original DataFrame
      |
      v
4 Partitions
      |
      v
repartition(8)
      |
      v
FULL SHUFFLE
      |
      v
8 New Partitions
```

Spark redistributes data across the requested partitions.

This can improve parallel processing when the previous partition count is insufficient.

### When should we use repartition()?

Scenario:

- Dataset contains 10 million records.
- Only 1 input partition exists.
- 8 CPU cores are available.
- A computationally expensive transformation follows.

Potential solution:

```python
df_parallel = df.repartition(8)
```

Now Spark can execute tasks across 8 partitions concurrently, assuming sufficient processing resources.

**Important:**

- 8 partitions do not mean 8 executors.
- More partitions do not guarantee 8x faster processing.
- Repartitioning introduces a shuffle.
- The performance benefit must outweigh the redistribution cost.

### RetailPulse Example

```python
# Bronze orders
df_orders = spark.table("bronze.orders")

# Increase processing partitions if justified
df_parallel = df_orders.repartition(8)

# Silver transformation
df_silver = df_parallel.withColumn(
    "order_value",
    F.col("order_value").cast("double")
)
```

---

## 3. coalesce(n) — Reduce Partitions

Syntax:

```python
df.coalesce(2)
```

### How it works

Suppose a DataFrame has 8 partitions.

```text
Original DataFrame
       |
       v
8 Partitions
       |
       v
coalesce(2)
       |
       v
2 Partitions
```

Spark combines existing partitions using a narrow dependency, normally without performing a full shuffle.

### When should we use coalesce()?

Scenario:

Our Gold aggregation produces only four rows:

```text
Delhi
Mumbai
Pune
Chennai
```

We want to write this small result as a standalone Parquet output.

Potential solution:

```python
df_gold = df_orders.groupBy("city").agg(
    F.sum("order_value").alias("total_revenue")
)

df_gold_single = df_gold.coalesce(1)

df_gold_single.write \
    .mode("overwrite") \
    .parquet(output_path)
```

A typical non-partitioned write produces one Parquet data file, although additional metadata files may exist.

### Important Considerations

- coalesce() normally avoids a full shuffle.
- It is primarily used for reducing partition counts.
- coalesce(1) restricts the resulting processing stage to one task.
- For large datasets, this can create a performance bottleneck.
- Do not automatically use coalesce(1) for every production write.

For production Delta tables, consider Databricks Optimized Writes, Auto Compaction and OPTIMIZE where applicable.

---

## 4. Practical Comparison

Assume our original DataFrame has 8 partitions.

| Code | Result | Full Shuffle |
|---|---:|---|
| df.repartition(16) | 16 partitions | Yes |
| df.repartition(4) | 4 partitions | Yes |
| df.coalesce(4) | 4 partitions | Normally No |
| df.coalesce(1) | 1 partition | Normally No |
| df.coalesce(16) | Does not increase beyond 8 | No |

### Important Rule

repartition() = Redistributes data through a shuffle.

coalesce() = Reduces partitions, normally without a full shuffle.

Neither operation is universally faster.

---

## 5. How to Decide in a Real Project

| Situation | What should we consider? |
|---|---|
| Too few partitions | Investigate repartition() |
| Insufficient parallelism | Investigate increasing partitions |
| Uneven data distribution | Consider redistribution and skew handling |
| Too many tiny output partitions | Consider reducing partitions |
| Small standalone output | coalesce() may be appropriate |
| Large production Delta table | Prefer workload-specific and Databricks-managed optimisation |
| Shuffle is already efficient | Avoid unnecessary repartitioning |

### Decision Process

```text
Check Existing Partition Count
             |
             v
Analyse Query Profile
             |
             v
Are tasks underutilising CPU resources?
             |
       YES --+--> Consider repartition()
             |
             v
Are there excessive small output partitions?
             |
       YES --+--> Consider coalesce()
             |
             v
Benchmark Execution
             |
             v
Keep the change only if it improves performance
```

---

## 6. Important: Do Not Confuse partitionBy()

These three methods solve different problems.

| Method | Purpose |
|---|---|
| repartition(n) | Redistribute Spark processing partitions |
| coalesce(n) | Reduce Spark processing partitions |
| write.partitionBy("order_date") | Organise files into storage directories |

Example:

```python
df.write \
    .partitionBy("order_date") \
    .format("delta") \
    .save(output_path)
```

This organises the written data using order-date partition values.

It is NOT the same as repartitioning DataFrame tasks.

---

## 7. Interview Questions

**Q1. What is the difference between repartition and coalesce?**

repartition() performs a shuffle to redistribute data and can increase or decrease partitions.

coalesce() primarily reduces partitions and normally avoids a full shuffle.

**Q2. Is repartition always slower than coalesce?**

No. repartition() introduces shuffle overhead, but better data distribution and parallelism may improve overall query performance.

**Q3. Can coalesce increase partitions?**

No. Use repartition() when you need to increase the partition count.

**Q4. Should we always use coalesce(1) before writing files?**

No. It may create a single-task bottleneck. It is appropriate only when the output size and write requirements justify it.

**Q5. Is repartition(8) always faster than repartition(200)?**

No. The correct partition count depends on workload, resources, data size, distribution and shuffle overhead.

---

## Final Learning Conclusion

**repartition() → Redistribute data; full shuffle; can increase or decrease partitions.**

**coalesce() → Reduce partitions; normally no full shuffle; may reduce parallelism.**

**partitionBy() → Organise data physically on storage using column values.**

### Golden Rule

Do not manually change partitions without a reason.

First inspect partition count, execution plan and Query Profile.

Then optimise based on actual workload behaviour rather than assumptions.